# Assignment 4 — N-gram Language Models (Gujarati)
**ILLM** · **U23AI051**

Same corpus as Assignment 3: the 1,000,000-sentence IndicCorpV2 Gujarati split, with the same
train / dev / test partition (dev and test are 1000 sentences each, drawn 100 per length decile).

We build **unigram, bigram, trigram and quadgram** language models (N = 1, 2, 3, 4) under five
settings and compare their test-set **perplexities**:

| # | Setting | Formula |
|---|---------|---------|
| 1 | Unsmoothed | raw counts, no smoothing |
| 2 | Add-one (Laplace) | $P(w\mid h)=\frac{C(h,w)+1}{C(h)+V}$ |
| 3 | Add-K (0 < K < 1) | $P(w\mid h)=\frac{C(h,w)+K}{C(h)+K\cdot V}$ |
| 4 | Interpolated | recursive linear interpolation, λ tuned on dev |
| 5 | Kneser-Ney | interpolated KN with discount $d=0.75$ |

For **unigrams** the interpolated formula is, as specified:
$$P(w)=\lambda\frac{C(w)}{N}+(1-\lambda)\frac{1}{V}$$
where λ is the interpolation constant and V is the vocabulary size.

## 0. Settings

In [1]:
import os, sys, time, math, json
from collections import Counter

sys.path.insert(0, os.path.join(os.getcwd(), "..", "common"))
from corpus import load_splits

ADD_K = 0.5        # add-K constant, chosen in (0, 1)
KN_D  = 0.75       # Kneser-Ney discount

UNK, S, EOS = "<unk>", "<s>", "</s>"          # special tokens
MAX_ORDER = 4

print("settings loaded")

C:\Users\KURUV PATEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


settings loaded


## 1. Corpus and vocabulary

Each sentence is whitespace-tokenized (as in the earlier assignments) and padded with
`order-1` `<s>` tokens plus a trailing `</s>`. Out-of-vocabulary test/dev words map to `<unk>`. 

In [2]:
train, dev, test = load_splits()
print(f"train {len(train):,}   dev {len(dev):,}   test {len(test):,}")


def build_vocabulary(train_sentences):
    types = sorted({w for s in train_sentences for w in s.split()})
    word_to_id = {w: i for i, w in enumerate([UNK, S, EOS] + types)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(sentences, word_to_id, padding):
    """Integer-encode with `padding` <s> tokens at the start and </s> at the end."""
    unk_id, s_id, eos_id = word_to_id[UNK], word_to_id[S], word_to_id[EOS]
    out = []
    for sent in sentences:
        ids = [s_id] * padding + [word_to_id.get(w, unk_id) for w in sent.split()] + [eos_id]
        out.append(ids)
    return out


word_to_id, id_to_word = build_vocabulary(train)
vocab_size = len(word_to_id) - 1          # predicted vocabulary: <unk>, </s>, + word types
print(f"vocabulary size V = {vocab_size:,}")

train 1,000,000   dev 1,000   test 1,000


vocabulary size V = 797,735


## 2. N-gram counts

Each order is counted from a corpus padded with exactly `order-1` `<s>` tokens. Context counts
$C(h)$ are then derived by summing the higher-order counts, so the `<s>` history is counted
correctly and never leaks into the predicted vocabulary.

In [3]:
def build_counts(encoded, order):
    counter = Counter()
    for ids in encoded:
        if len(ids) >= order:
            counter.update(tuple(ids[i:i + order]) for i in range(len(ids) - order + 1))
    return counter


def context_counts(counts, order):
    """(order-1)-gram context -> count, by summing the `order`-gram counts."""
    cc = Counter()
    for seq, value in counts[order].items():
        cc[seq[:-1]] += value
    return cc


def build_models(train_sentences, word_to_id, max_order):
    counts, ctx = {}, {}
    for order in range(1, max_order + 1):
        t0 = time.time()
        enc = encode(train_sentences, word_to_id, order - 1)
        counts[order] = build_counts(enc, order)
        if order >= 2:
            ctx[order] = context_counts(counts, order)
        print(f"  order {order}: {len(counts[order]):,} distinct {order}-grams "
              f"({time.time() - t0:.0f}s)")
    return counts, ctx


counts, ctx = build_models(train, word_to_id, MAX_ORDER)
total_unigram = sum(counts[1].values())
print(f"total unigram tokens = {total_unigram:,}")

  order 1: 797,734 distinct 1-grams (56s)


  order 2: 6,288,523 distinct 2-grams (63s)


  order 3: 11,326,326 distinct 3-grams (51s)


  order 4: 13,463,693 distinct 4-grams (55s)
total unigram tokens = 16,117,528


## 3. MLE and perplexity

Perplexity is $\mathrm{PP}=\exp\!\left(-\frac{1}{M}\sum_{i}\ln P(w_i\mid h_i)\right)$,
where $M$ is the number of predicted tokens in the evaluation set (words plus `</s>`).

In [4]:
def mle(counts, ctx, order, word, history, total_unigram):
    if order == 1:
        return counts[1].get((word,), 0) / total_unigram
    context = tuple(history[-(order - 1):])
    num = counts[order].get(context + (word,), 0)
    den = ctx[order].get(context, 0)
    return num / den if den else 0.0


def perplexity(log_probs, m):
    return math.exp(-sum(log_probs) / m) if m else float("inf")

## 4. Kneser-Ney smoothing ($d=0.75$)

Interpolated Kneser-Ney. The **highest order** uses raw counts with absolute discounting; every
lower order (down to the unigram) uses **continuation counts** — the number of distinct words
that precede a sequence — so frequent-but-uninformative words are down-weighted.

$$P_{KN}(w\mid h)=\frac{\max(C(h,w)-d,\,0)}{C(h)}+\frac{d}{C(h)}\cdot N_{1+}(h\,\cdot)\,P_{KN}(w\mid h')$$

The unigram base is the continuation distribution $P_{cont}(w)=N_{1+}(\bullet\, w)/N_{1+}(\bullet\,\bullet)$.
Out-of-vocabulary words (mapped to `<unk>`) receive a floor of $d$ continuation mass so the
perplexity stays finite.

In [5]:
def build_continuation(counts, ctx, max_order):
    cont = {}
    for k in range(1, max_order):
        cont[k] = Counter()
        for seq in counts[k + 1]:
            cont[k][seq[1:]] += 1                     # distinct left words for tail seq

    cont_denom, cont_followers = {}, {}
    for o in range(2, max_order):
        denom, followers = Counter(), Counter()
        for seq, value in cont[o].items():
            context = seq[:-1]
            denom[context] += value
            followers[context] += 1
        cont_denom[o] = denom
        cont_followers[o] = followers

    raw_followers = Counter()
    for seq in counts[max_order]:
        raw_followers[seq[:-1]] += 1

    return {"cont": cont, "cont_denom": cont_denom, "cont_followers": cont_followers,
            "raw_followers": raw_followers,
            "total_bigram_types": sum(cont[1].values()),
            "counts": counts, "ctx": ctx, "max_order": max_order}


tables = build_continuation(counts, ctx, MAX_ORDER)
print("continuation tables built")


def kn_prob(tables, word, history, order):
    counts, ctx = tables["counts"], tables["ctx"]
    cont = tables["cont"]
    d, n = KN_D, tables["max_order"]

    if order == 1:
        total = tables["total_bigram_types"]
        if total == 0:
            return 0.0
        c = cont[1].get((word,), 0)
        if c == 0:
            c = d          # floor for out-of-vocabulary words (mapped to <unk>)
        return c / total

    context = tuple(history[-(order - 1):])
    seq = context + (word,)
    lower = kn_prob(tables, word, history[1:] if len(history) > 1 else (), order - 1)

    if order == n:
        c_hw = counts[order].get(seq, 0)
        c_h = ctx[order].get(context, 0)
        followers = tables["raw_followers"].get(context, 0)
        if c_h == 0:
            return lower
        return max(c_hw - d, 0) / c_h + (d / c_h) * followers * lower
    else:
        cont_hw = cont[order].get(seq, 0)
        denom = tables["cont_denom"][order].get(context, 0)
        followers = tables["cont_followers"][order].get(context, 0)
        if denom == 0:
            return lower
        return max(cont_hw - d, 0) / denom + (d / denom) * followers * lower

continuation tables built


## 5. Interpolated smoothing

Recursive linear interpolation, one λ per level, tuned greedily on the dev set (lower levels
are fixed before the next level is tuned):

$$P_{int}^{(n)}(w\mid h)=\lambda_n\,P_{ML}^{(n)}(w\mid h)+(1-\lambda_n)\,P_{int}^{(n-1)}(w\mid h')$$

with the unigram base $P_{int}^{(1)}(w)=\lambda_1\frac{C(w)}{N}+(1-\lambda_1)\frac{1}{V}$.

In [6]:
def interpolated_prob(counts, ctx, order, word, history, lambdas, vocab_size, total_unigram):
    lam = lambdas[order]
    if order == 1:
        return lam * mle(counts, ctx, 1, word, (), total_unigram) + (1 - lam) / vocab_size
    return (lam * mle(counts, ctx, order, word, history, total_unigram)
            + (1 - lam) * interpolated_prob(counts, ctx, order - 1, word,
                                            history[1:], lambdas, vocab_size, total_unigram))

## 6. One scorer for every (order, smoothing) combination

In [7]:
def score_smoothing(counts, ctx, order, encoded, smoothing, vocab_size,
                    lam=None, tables=None, total_unigram=None):
    log_probs = []

    for ids in encoded:
        for i in range(order - 1, len(ids)):
            word = ids[i]
            history = tuple(ids[i - (order - 1):i])

            if smoothing == "unsmoothed":
                p = mle(counts, ctx, order, word, history, total_unigram)
                log_probs.append(math.log(p) if p > 0 else -math.inf)

            elif smoothing in ("add1", "addk"):
                k = 1.0 if smoothing == "add1" else ADD_K
                if order == 1:
                    c = counts[1].get((word,), 0)
                    p = (c + k) / (total_unigram + k * vocab_size)
                else:
                    c = counts[order].get(history + (word,), 0)
                    c_context = ctx[order].get(history, 0)
                    p = (c + k) / (c_context + k * vocab_size)
                log_probs.append(math.log(p))

            elif smoothing == "interpolated":
                p = interpolated_prob(counts, ctx, order, word, history,
                                      lam, vocab_size, total_unigram)
                log_probs.append(math.log(p))

            elif smoothing == "kneser_ney":
                p = kn_prob(tables, word, history, order)
                log_probs.append(math.log(p) if p > 0 else -math.inf)

            else:
                raise ValueError(smoothing)

    m = sum(len(ids) - (order - 1) for ids in encoded)
    return perplexity(log_probs, m), m

## 7. Tune λ on dev, then evaluate on test

In [8]:
print("tuning interpolation lambdas on dev (greedy, per level)...")
lambdas = {}
for order in range(1, MAX_ORDER + 1):
    dev_enc = encode(dev, word_to_id, order - 1)
    best_lam, best_pp = None, math.inf
    for lam in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        lambdas[order] = lam
        pp, _ = score_smoothing(counts, ctx, order, dev_enc, "interpolated",
                                vocab_size, lam=lambdas, total_unigram=total_unigram)
        if pp < best_pp:
            best_pp, best_lam = pp, lam
    lambdas[order] = best_lam
    print(f"  order {order}: lambda = {best_lam}  (dev perplexity {best_pp:,.1f})")


print("\n=== TEST PERPLEXITIES ===")
smoothings = ["unsmoothed", "add1", "addk", "interpolated", "kneser_ney"]
print(f"{'order':<9}" + "".join(f"{s:>14}" for s in smoothings))

results = []
for order in range(1, MAX_ORDER + 1):
    test_enc = encode(test, word_to_id, order - 1)
    name = {1: "unigram", 2: "bigram", 3: "trigram", 4: "quadgram"}[order]
    row = {"order": name}
    line = f"{name:<9}"
    for s in smoothings:
        kwargs = {"total_unigram": total_unigram}
        if s == "interpolated":
            kwargs["lam"] = lambdas
        if s == "kneser_ney":
            kwargs["tables"] = tables
        pp, _ = score_smoothing(counts, ctx, order, test_enc, s, vocab_size, **kwargs)
        row[s] = pp
        line += f"{('inf' if math.isinf(pp) else f'{pp:,.1f}'):>14}"
    results.append(row)
    print(line)

tuning interpolation lambdas on dev (greedy, per level)...


  order 1: lambda = 0.9  (dev perplexity 7,697.4)


  order 2: lambda = 0.6  (dev perplexity 1,426.7)


  order 3: lambda = 0.1  (dev perplexity 1,319.7)


  order 4: lambda = 0.1  (dev perplexity 1,366.8)

=== TEST PERPLEXITIES ===
order        unsmoothed          add1          addk  interpolated    kneser_ney


unigram             inf       7,130.8       7,181.0       7,224.5       9,662.9


bigram              inf      40,578.3      27,695.2       1,381.7       1,251.9


trigram             inf     244,110.3     202,728.6       1,277.5       1,072.7


quadgram            inf     424,393.2     388,192.5       1,328.6         981.5


## 8. Results (from this run)

| order | unsmoothed | add-1 | add-K (K=0.5) | interpolated | Kneser-Ney |
|-------|-----------|-------|---------------|--------------|------------|
| unigram | ∞ | 7,130.8 | 7,181.0 | 7,224.5 | 9,662.9 |
| bigram | ∞ | 40,578.3 | 27,695.2 | 1,381.7 | 1,251.9 |
| trigram | ∞ | 244,110.3 | 202,728.6 | 1,277.5 | 1,072.7 |
| quadgram | ∞ | 424,393.2 | 388,192.5 | 1,328.6 | **981.5** |

**Reading the table**

* **Unsmoothed is ∞ everywhere.** Any unseen n-gram gets probability 0, so a single OOV word
  (3.3% of test tokens) or unseen n-gram drives $\ln P$ to $-\infty$. That is the whole reason
  smoothing exists.
* **Add-1 and add-K get *worse* with order**, not better. The vocabulary is huge
  ($V \approx 797{,}735$), so the $+K$ mass added to every unseen n-gram is spread across an
  enormous denominator. The result is severe over-smoothing — by the quadgram the model is
  barely better than chance, and Laplace is the worst of all.
* **Interpolation improves until the trigram** (1,277.5) then flattens at the quadgram (1,328.6):
  the quadgram is sparse, so its λ drops to 0.1 and the model leans on the lower orders.
* **Kneser-Ney is monotonically better with order** and wins overall: **981.5** at the quadgram.
  The continuation counts stop high-frequency-but-uninformative words from dominating the lower
  orders, which is exactly the failure mode that makes plain interpolation flatline.
* **Best model: Kneser-Ney quadgram (981.5)** — about 25% better than the best interpolated model
  (1,277.5) and dramatically better than the unigram baseline.